<a href="https://colab.research.google.com/github/hsamala688/Space_Weather_Predictor/blob/main/tec_sfno_residual_realdata_latest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TEC Forecasting — SFNO Model (Residual TEC )

Spherical Fourier Neural Operator for global ionospheric TEC forecasting.

**What this notebook does:**
Builds and trains the full SFNO model against fake residual TEC data.
When Person 1 delivers real data, swap `FakeTECDataset` for their `Dataset` — everything else stays the same.

**Architecture:**
- GRU encoder reads OMNI solar wind → context vector → initializes SphericalConvGRU hidden state
- SphericalConvGRU processes TEC residual history on Gauss-Legendre grid
- 4 SFNO blocks (64 → 128 → 128 → 64 channels)
- Pointwise linear output head → t+1h, t+2h, t+3h forecasts
- Storm-weighted MSE + Sobolev regularization loss
- Scheduled teacher forcing (Bengio et al. 2015)
- W&B logging + checkpointing

**TEC mode: RESIDUAL**
Model predicts (TEC - climatology). At inference, add climatology back for real TECU values and PyVista visualization. Climatology is zeros placeholder until Person 1 delivers real tensor.

**Grid decisions (locked):**
- Gauss-Legendre quadrature (Person 2 confirmed)
- Longitude: 0 to 360
- Cell-centered (never includes poles)
- float32 throughout
- H = L_max, W = 2 × L_max (placeholders — swap when Person 2 delivers L_max)

**Cells:**
1. Install libraries
2. Imports and GPU check
3. Config
4. Grid checker utility
5. Fake dataset (residual TEC)
6. Gauss-Legendre area weights
7. Sobolev regularization weights
8. SFNO Block
9. SphericalConvGRU Cell
10. Full TEC SFNO Model
11. Loss function
12. Teacher forcing schedule
13. Evaluation metrics
14. Training loop
15. Autoregressive inference loop
16. Baselines
17. Evaluate baselines
18. Checkpointing
19. Main — run everything

## Cell 0 — Setup Environment

If your data is in another location, change !cp string to your specific data storage location.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/tec_data/falisha_windows_gl23x45.tar.gz" .
!tar -xzf falisha_windows_gl23x45.tar.gz

import os
os.makedirs('data_pull', exist_ok=True)
os.rename('falisha_dataset.py', 'data_pull/falisha_dataset.py')

Mounted at /content/drive
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'


## Cell 1 — Install libraries


In [ ]:
!pip install torch-harmonics wandb
print('Done.')

Done.


## Cell 2 — Imports and GPU check

If you see 'No GPU detected', go to Runtime → Change runtime type → T4 GPU.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import wandb

from torch_harmonics import RealSHT, InverseRealSHT
from torch_harmonics.quadrature import legendre_gauss_weights

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cpu':
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU')

print("this works")

Using device: cuda
this works


## Cell 3 — Config

All hyperparameters and grid settings in one place.
Swap H, W, l_max, m_max when Person 2 delivers L_max. Rule: H = L_max, W = 2 × L_max.

In [ ]:
CONFIG = {
    # Grid
    # Grid: Gauss-Legendre (Person 2 confirmed). Longitude: 0 to 360.
    'H':     23,   # latitude points
    'W':     45,  # longitude points
    'l_max': 23,   # Person 2 derived
    'm_max': 23,   # same as l_max

    # Data
    'n_timesteps':     6,   # hours of TEC history
    'n_omni_features': 6,   # b_magnitude, by_gsm, bz_gsm, flow_speed, proton_density, kp_3hour
    'n_horizons':      3,   # t+1h, t+2h, t+3h

    # TEC mode: residual
    # Person 1 delivers (TEC - climatology). At inference add climatology back.

    # Model
    'gru_hidden_size': 128,
    'sfno_channels':   [64, 128, 128, 64],
    'n_sfno_blocks':   4,

    # Training
    'batch_size':     64,
    'learning_rate':  1e-4,
    'n_epochs':       20,
    'lambda_sobolev': 1e-7,   # tune after baseline results
    'n_kp_bins':      18,     # Kp storm-weight histogram bins

    # W&B
    'wandb_project':  'tec-sfno',
    'wandb_run_name': 'residual-real-data-run-1',

    # Drive Link
    'data_root': 'data/falisha_windows_gl23x45'

}

print('Config loaded.')
print(f"Grid: {CONFIG['H']}x{CONFIG['W']} | L_max: {CONFIG['l_max']} ")

Config loaded.
Grid: 23x45 | L_max: 23 


## Cell 4 — Grid checker utility

Run this to see the exact Gauss-Legendre latitude and longitude vectors.
This answers Person 1's question: 'what are the literal coordinate lists?'
Also confirms whether grid is cell-centered (expected) or poles-inclusive.

In [ ]:
def check_grid(H, W):
    cos_lats, _ = legendre_gauss_weights(H)
    lats_deg    = np.degrees(np.arccos(cos_lats.numpy())) - 90
    lons_deg    = np.linspace(0, 360, W, endpoint=False)

    print(f'Gauss-Legendre grid  H={H}, W={W}:')
    print(f'  lat first 3: {lats_deg[:3].round(4)}')
    print(f'  lat last  3: {lats_deg[-3:].round(4)}')
    print(f'  lon first 3: {lons_deg[:3].round(4)}')
    print(f'  lon last  3: {lons_deg[-3:].round(4)}')
    poles = np.isclose(abs(lats_deg[0]), 90) or np.isclose(abs(lats_deg[-1]), 90)
    print(f'  Poles included: {poles}  (expected: False = cell-centered)')
    return lats_deg, lons_deg

# Run to get exact vectors for Person 1:
check_grid(CONFIG['H'],CONFIG['W'])

Gauss-Legendre grid  H=23, W=45:
  lat first 3: [84.1372 76.5424 68.9028]
  lat last  3: [-68.9028 -76.5424 -84.1372]
  lon first 3: [ 0.  8. 16.]
  lon last  3: [336. 344. 352.]
  Poles included: False  (expected: False = cell-centered)


(array([ 84.13719338,  76.5424155 ,  68.90279369,  61.25304995,
         53.59947903,  45.94407041,  38.28765113,  30.63062914,
         22.97323216,  15.31560115,   7.65783255,   0.        ,
         -7.65783255, -15.31560115, -22.97323216, -30.63062914,
        -38.28765113, -45.94407041, -53.59947903, -61.25304995,
        -68.90279369, -76.5424155 , -84.13719338]),
 array([  0.,   8.,  16.,  24.,  32.,  40.,  48.,  56.,  64.,  72.,  80.,
         88.,  96., 104., 112., 120., 128., 136., 144., 152., 160., 168.,
        176., 184., 192., 200., 208., 216., 224., 232., 240., 248., 256.,
        264., 272., 280., 288., 296., 304., 312., 320., 328., 336., 344.,
        352.]))

## Cell 5 — Dataset (residual TEC)

Produces tensors in the exact shapes Person 1 will deliver.
TEC values are zero-centered (~2 TECU std) because they represent anomalies above climatology.

When Person 1's real Dataset arrives, swap this class out — all shapes stay identical.

To test learnability (check model can overfit one batch), uncomment the deterministic target line.

In [ ]:
from data_pull.falisha_dataset import make_falisha_dataloader, FalishaDTECDataset

def make_dataloader(config, split, shuffle=None):
    if shuffle is None:
        shuffle = (split == "train")
    return make_falisha_dataloader(
        root=config['data_root'],
        split=split,
        batch_size=config['batch_size'],
        shuffle=shuffle,
    )

# Quick shape check
_loader = make_dataloader(CONFIG, split="train")
_batch = next(iter(_loader))
print(_batch["tec_input"].shape, _batch["omni_input"].shape, _batch["target"].shape)

torch.Size([4, 6, 23, 45]) torch.Size([4, 6, 6]) torch.Size([4, 3, 23, 45])


## Cell 6 — Gauss-Legendre area weights

The equiangular grid over-represents poles (more pixels per area).
Gauss-Legendre quadrature weights are the mathematically correct area weights
for this grid — they already account for the sphere's area element.
Used in both the loss function and RMSE evaluation.

In [ ]:
def make_gl_weights(H, device):
    """
    Returns: [1, 1, H, 1] — broadcasts over [B, horizons, H, W]
    """
    _, w = legendre_gauss_weights(H)
    w    = w.to(torch.float32).to(device)
    w    = w / w.sum()   # normalize so weights sum to 1
    return w.view(1, 1, H, 1)


gl_weights = make_gl_weights(CONFIG['H'], DEVICE)
print(f'GL weights shape: {gl_weights.shape}')
print(f'Weights sum to:   {gl_weights.sum().item():.4f}  (expected 1.0)')

GL weights shape: torch.Size([1, 1, 23, 1])
Weights sum to:   1.0000  (expected 1.0)


## Cell 7 — Sobolev regularization weights

H^1(S^2) penalty: eigenvalues of the Laplace-Beltrami operator on S^2.
Formula: (1 + l*(l+1)) per spherical harmonic degree l.
Person 2 derives this analytically — we use the formula directly.
Added to the loss to penalize high-frequency noise in predictions.

In [ ]:
def make_sobolev_weights(l_max, device):
    """
    Returns: [l_max]
    """
    l = torch.arange(l_max, dtype=torch.float32, device=device)
    return 1.0 + l * (l + 1.0)


sobolev_weights = make_sobolev_weights(CONFIG['l_max'], DEVICE)
print(f'Sobolev weights shape: {sobolev_weights.shape}')
print(f'First few values (l=0,1,2,3): {sobolev_weights[:4].tolist()}')
# l=0 → 1.0, l=1 → 3.0, l=2 → 7.0, l=3 → 13.0 — penalizes high freq more

Sobolev weights shape: torch.Size([23])
First few values (l=0,1,2,3): [1.0, 3.0, 7.0, 13.0]


## Cell 7b — Storm weights (Deliverable 4)

Inverse-density weighting on Kp. Built once from real training data
(last input timestep's Kp, `omni_input[:, -1, 5]`), fit on train split only.

In [ ]:
def build_storm_weight_table(kp_values, n_bins):
    """
    kp_values: 1D array of Kp (TRAINING split only, last input timestep).
    Returns: (bin_edges, weights) — weights normalized to mean 1.
    Bin range comes from the data's own min/max (values are normalized Kp,
    not raw 0-9 — inverse-density weighting is unaffected by the linear
    z-score rescaling, only the bin edge labels are in normalized units).
    """
    kp_values = np.asarray(kp_values)
    bin_edges = np.linspace(kp_values.min(), kp_values.max(), n_bins + 1)
    counts, _ = np.histogram(kp_values, bins=bin_edges)
    density   = counts / counts.sum()
    density   = np.clip(density, 1e-8, None)   # avoid div-by-zero on empty bins
    weights   = 1.0 / density
    weights   = weights / weights.mean()       # normalize so avg weight = 1
    return torch.tensor(bin_edges, dtype=torch.float32), torch.tensor(weights, dtype=torch.float32)


def lookup_storm_weight(kp_batch, bin_edges, weights):
    """kp_batch: [B] -> [B] storm weights, via the precomputed table."""
    bin_idx = torch.bucketize(kp_batch, bin_edges[1:-1])
    return weights.to(kp_batch.device)[bin_idx]


# Build the table once from REAL training Kp — read directly off the
# memory-mapped array (fast), no need to iterate the full DataLoader.
_train_ds_for_kp = FalishaDTECDataset(root=CONFIG['data_root'], split='train')
_train_kp_last    = np.array(_train_ds_for_kp.omni_input[:, -1, 5])  # last timestep, Kp = feature index 5
storm_bin_edges, storm_weights_table = build_storm_weight_table(_train_kp_last, CONFIG['n_kp_bins'])
del _train_ds_for_kp, _train_kp_last

print(f'Storm weight table: {CONFIG["n_kp_bins"]} bins from real training Kp')
print(f'Weights range: [{storm_weights_table.min():.2f}, {storm_weights_table.max():.2f}]')


Storm weight table: 18 bins from real training Kp
Weights range: [0.01, 6.32]


## Cell 8 — SFNO Block

The core building block. Each block does:
1. Forward SHT: pixel space → spherical harmonic coefficients
2. Multiply by learnable complex weights (the convolution in frequency space)
3. Inverse SHT: coefficients → pixel space
4. GELU activation + residual connection

This is Person 2's SphericalConvLayer — same math, implemented here.

In [ ]:
class SFNOBlock(nn.Module):
    def __init__(self, in_channels, out_channels, H, W, l_max, m_max):
        super().__init__()
        # Gauss-Legendre grid throughout (Person 2 confirmed)
        self.sht  = RealSHT(H, W, lmax=l_max, mmax=m_max, grid='legendre-gauss')
        self.isht = InverseRealSHT(H, W, lmax=l_max, mmax=m_max, grid='legendre-gauss')

        # Learnable spectral filter: real + imaginary stored separately
        self.filter_re = nn.Parameter(
            torch.randn(out_channels, in_channels, l_max, m_max) * 0.02
        )
        self.filter_im = nn.Parameter(
            torch.randn(out_channels, in_channels, l_max, m_max) * 0.02
        )

        # Deliverable 3 — Gibbs window (Fejer), Saksham's formula:
        # sigma_l = 1 - l/(L_max+1), l=0..L_max. torch-harmonics' l_max kwarg
        # is already the count (L_max+1=23), so sigma = 1 - arange(l_max)/l_max.
        # Fixed, not learned. Lives here (not in the top-level model forward())
        # because this is the only place the SHT/filter/inverse-SHT triple exists.
        l_idx = torch.arange(l_max, dtype=torch.float32)
        sigma = 1.0 - l_idx / l_max
        self.register_buffer('sigma_l', sigma.view(1, 1, -1, 1))  # broadcast [B,ch,l,m]

        # Pointwise residual projection (handles channel dim change)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.act       = nn.GELU()

    def forward(self, x):
        """x: [B, in_ch, H, W] -> [B, out_ch, H, W]"""
        residual   = self.pointwise(x)
        x_hat      = self.sht(x)                                    # -> complex [B, in_ch, l, m]
        W_c        = torch.complex(self.filter_re, self.filter_im)  # complex filter
        x_filtered = torch.einsum('bilm,oilm->bolm', x_hat, W_c)    # spectral multiply
        x_filtered = x_filtered * self.sigma_l                      # Gibbs window
        x_out      = self.isht(x_filtered)                          # -> [B, out_ch, H, W]
        return self.act(x_out + residual)


# Shape test
_block = SFNOBlock(1, 16, CONFIG['H'], CONFIG['W'], CONFIG['l_max'], CONFIG['m_max']).to(DEVICE)
_x     = torch.randn(2, 1, CONFIG['H'], CONFIG['W']).to(DEVICE)
_out   = _block(_x)
print(f'SFNOBlock input:  {_x.shape}')
print(f'SFNOBlock output: {_out.shape}   (expected [2, 16, {CONFIG["H"]}, {CONFIG["W"]}])')


SFNOBlock input:  torch.Size([2, 1, 23, 45])
SFNOBlock output: torch.Size([2, 16, 23, 45])   (expected [2, 16, 23, 45])


## Cell 9 — SphericalConvGRU Cell

A GRU cell where the matrix multiplications are replaced with spherical convolutions.
The hidden state lives on the sphere: [B, hidden_channels, H, W].
This is what processes the TEC history timestep by timestep.

In [ ]:
class SphericalConvGRUCell(nn.Module):
    def __init__(self, input_channels, hidden_channels, H, W, l_max, m_max):
        super().__init__()
        self.hidden_channels = hidden_channels
        combined = input_channels + hidden_channels

        self.reset_gate  = SFNOBlock(combined, hidden_channels, H, W, l_max, m_max)
        self.update_gate = SFNOBlock(combined, hidden_channels, H, W, l_max, m_max)
        self.candidate   = SFNOBlock(combined, hidden_channels, H, W, l_max, m_max)

    def forward(self, x_t, h_prev):
        """
        x_t:    [B, input_ch,  H, W]  — TEC residual at current timestep
        h_prev: [B, hidden_ch, H, W]  — previous hidden state on sphere
        → h_new [B, hidden_ch, H, W]
        """
        cat_xh  = torch.cat([x_t, h_prev], dim=1)
        r       = torch.sigmoid(self.reset_gate(cat_xh))
        z       = torch.sigmoid(self.update_gate(cat_xh))
        cat_xrh = torch.cat([x_t, r * h_prev], dim=1)
        h_cand  = torch.tanh(self.candidate(cat_xrh))
        return (1 - z) * h_prev + z * h_cand


# Shape test
_gru_cell = SphericalConvGRUCell(
    input_channels=1, hidden_channels=CONFIG['sfno_channels'][0],
    H=CONFIG['H'], W=CONFIG['W'],
    l_max=CONFIG['l_max'], m_max=CONFIG['m_max']
).to(DEVICE)
_x_t   = torch.randn(2, 1, CONFIG['H'], CONFIG['W']).to(DEVICE)
_h     = torch.zeros(2, CONFIG['sfno_channels'][0], CONFIG['H'], CONFIG['W']).to(DEVICE)
_h_new = _gru_cell(_x_t, _h)
print(f'GRU input:   {_x_t.shape}')
print(f'GRU h_prev:  {_h.shape}')
print(f'GRU h_new:   {_h_new.shape}  (expected same as h_prev)')

GRU input:   torch.Size([2, 1, 23, 45])
GRU h_prev:  torch.Size([2, 64, 23, 45])
GRU h_new:   torch.Size([2, 64, 23, 45])  (expected same as h_prev)


## Cell 10 — Full TEC SFNO Model

Puts everything together:
1. OMNI GRU encoder → context vector → initializes SphericalConvGRU hidden state
2. SphericalConvGRU processes TEC residual history (6 timesteps)
3. Spectral window (identity placeholder — Person 2 delivers σ_l formula)
4. 4 SFNO blocks (64 → 128 → 128 → 64 channels)
5. Pointwise output head → 3 forecast horizons

Input:  tec_input [B, T, H, W] + omni_input [B, T, 5]
Output: [B, 3, H, W] — predicted TEC residuals at t+1h, t+2h, t+3h

In [ ]:
class TECSFNOModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        H        = config['H']
        W        = config['W']
        l_max    = config['l_max']
        m_max    = config['m_max']
        gru_h    = config['gru_hidden_size']
        channels = config['sfno_channels']   # [64, 128, 128, 64]
        n_omni   = config['n_omni_features']
        n_horiz  = config['n_horizons']
        self.channels = channels

        # 1. OMNI GRU encoder
        self.omni_gru     = nn.GRU(n_omni, gru_h, batch_first=True)
        self.context_proj = nn.Linear(gru_h, channels[0])

        # 2. SphericalConvGRU
        self.sph_gru = SphericalConvGRUCell(
            input_channels=1, hidden_channels=channels[0],
            H=H, W=W, l_max=l_max, m_max=m_max
        )

        # 3. Gibbs window now lives inside each SFNOBlock (Cell 8) -- that's
        #    the only place the SHT -> filter -> inverse-SHT triple exists.

        # 4. 4 SFNO blocks
        ins  = [channels[0], channels[1], channels[2], channels[3]]
        outs = [channels[1], channels[2], channels[3], channels[3]]
        self.sfno_blocks = nn.ModuleList([
            SFNOBlock(ins[i], outs[i], H, W, l_max, m_max)
            for i in range(config['n_sfno_blocks'])
        ])

        # 5. Pointwise output head
        self.output_head = nn.Conv2d(channels[-1], n_horiz, kernel_size=1)

    def forward(self, tec_input, omni_input):
        """
        tec_input:  [B, T, H, W]  — TEC residuals
        omni_input: [B, T, 6]     — solar wind
        returns:    [B, 3, H, W]  — predicted residuals at t+1h, t+2h, t+3h
        """
        B, T, H, W = tec_input.shape

        # Step 1: OMNI GRU → initial hidden state
        _, h_n  = self.omni_gru(omni_input)     # h_n: [1, B, gru_h]
        context = h_n.squeeze(0)                 # [B, gru_h]
        h_init  = self.context_proj(context)     # [B, channels[0]]
        h = h_init.unsqueeze(-1).unsqueeze(-1).expand(
            B, self.channels[0], H, W
        ).contiguous()                           # [B, channels[0], H, W]

        # Step 2: SphericalConvGRU over TEC history
        for t in range(T):
            x_t = tec_input[:, t].unsqueeze(1)  # [B, 1, H, W]
            h   = self.sph_gru(x_t, h)

        # Step 3: 4 SFNO blocks (each applies its own Gibbs window internally)
        x = h

        for block in self.sfno_blocks:
            x = block(x)

        # Step 4: Output head
        return self.output_head(x)               # [B, 3, H, W]


# Shape test
model    = TECSFNOModel(CONFIG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')

_tec_in  = torch.randn(2, CONFIG['n_timesteps'], CONFIG['H'], CONFIG['W']).to(DEVICE)
_omni_in = torch.randn(2, CONFIG['n_timesteps'], CONFIG['n_omni_features']).to(DEVICE)
_pred    = model(_tec_in, _omni_in)
print(f'Model input  tec:  {_tec_in.shape}')
print(f'Model input  omni: {_omni_in.shape}')
print(f'Model output:      {_pred.shape}   (expected [2, 3, {CONFIG["H"]}, {CONFIG["W"]}])')

Model parameters: 52,316,547
Model input  tec:  torch.Size([2, 6, 23, 45])
Model input  omni: torch.Size([2, 6, 6])
Model output:      torch.Size([2, 3, 23, 45])   (expected [2, 3, 23, 45])


## Cell 11 — Loss function

Two terms:
1. **Storm-weighted, area-weighted MSE** — larger weight during geomagnetic storms.
   Storm weights are ones placeholder until Person 2 delivers the inverse-density formula.
   GL quadrature weights correct for unequal pixel areas on the equiangular grid.
2. **Sobolev H^1(S^2) proxy** — penalizes high-frequency spatial noise in predictions.
   Full spectral version comes from Person 2. Finite-difference proxy for now.

In [ ]:
def sobolev_penalty(model, sobolev_weights):
    """
    Deliverable 5. Penalizes the LEARNED FILTER WEIGHTS, not predictions.
    L_reg = sum over l of (1 + l(l+1)) * ||filter_re + i*filter_im||^2 at that l,
    summed over every SFNOBlock in the model.
    sobolev_weights: [l_max] -> reshaped to broadcast over [out,in,l,m]
    """
    w = sobolev_weights.view(1, 1, -1, 1)   # [1,1,l,1]
    total = 0.0
    for block in model.sfno_blocks:
        mag_sq = block.filter_re ** 2 + block.filter_im ** 2   # [out,in,l,m]
        total  = total + (mag_sq * w).sum()
    return total


def tec_loss(pred, target, gl_weights, model, sobolev_weights, storm_weights, lambda_sob):
    """
    pred / target:  [B, n_horizons, H, W]  -- residual TEC
    gl_weights:     [1, 1, H, 1]
    model:          needed to reach the SFNOBlock filter weights for Deliverable 5
    storm_weights:  [B]  -- from lookup_storm_weight(), Deliverable 4
    """
    # Term 1: Storm-weighted, area-weighted MSE
    sq_err         = (pred - target) ** 2 * gl_weights
    mse_per_sample = sq_err.mean(dim=[1, 2, 3])        # [B]
    w              = storm_weights / storm_weights.sum()
    weighted_mse   = (w * mse_per_sample).sum()

    # Term 2: Sobolev H^1(S^2) penalty on filter weights (Deliverable 5)
    sobolev_term = sobolev_penalty(model, sobolev_weights)

    loss = weighted_mse + lambda_sob * sobolev_term
    return loss, weighted_mse.item(), sobolev_term.item()


# Quick sanity check -- everything on DEVICE, matching where `model` lives
_pred    = torch.randn(4, 3, CONFIG['H'], CONFIG['W']).to(DEVICE)
_target  = torch.randn(4, 3, CONFIG['H'], CONFIG['W']).to(DEVICE)
_kp_fake = torch.randn(4).to(DEVICE)  # dummy for the sanity check only
_sw      = lookup_storm_weight(_kp_fake, storm_bin_edges.to(DEVICE), storm_weights_table.to(DEVICE))
_loss, _mse, _sob = tec_loss(_pred, _target, gl_weights, model, sobolev_weights, _sw, CONFIG['lambda_sobolev'])
print(f'Loss: {_loss:.4f}  (MSE: {_mse:.4f}, Sobolev: {_sob:.4f})')

Loss: 276.1494  (MSE: 0.0882, Sobolev: 2760612.0000)


## Cell 12 — Teacher forcing schedule

Linear decay from 1.0 → 0.0 over training (Bengio et al. 2015).
Epoch 0: always use ground truth input (stable early training).
Last epoch: always use model's own predictions (learns to handle its own errors).

In [ ]:
def teacher_forcing_prob(epoch, total_epochs):
    return max(0.0, 1.0 - (epoch / total_epochs))


# Visualize the schedule
probs = [teacher_forcing_prob(e, CONFIG['n_epochs']) for e in range(CONFIG['n_epochs'])]
print('Teacher forcing schedule:')
for e, p in enumerate(probs):
    bar = '█' * int(p * 20)
    print(f'  Epoch {e:02d}: {p:.2f} {bar}')

Teacher forcing schedule:
  Epoch 00: 1.00 ████████████████████
  Epoch 01: 0.95 ███████████████████
  Epoch 02: 0.90 ██████████████████
  Epoch 03: 0.85 █████████████████
  Epoch 04: 0.80 ████████████████
  Epoch 05: 0.75 ███████████████
  Epoch 06: 0.70 ██████████████
  Epoch 07: 0.65 █████████████
  Epoch 08: 0.60 ████████████
  Epoch 09: 0.55 ███████████
  Epoch 10: 0.50 ██████████
  Epoch 11: 0.45 █████████
  Epoch 12: 0.40 ████████
  Epoch 13: 0.35 ███████
  Epoch 14: 0.30 ██████
  Epoch 15: 0.25 █████
  Epoch 16: 0.20 ███
  Epoch 17: 0.15 ███
  Epoch 18: 0.10 █
  Epoch 19: 0.05 █


## Cell 13 — Evaluation metrics

Area-weighted RMSE per forecast horizon.

**Important:** always call this with real TECU values (after adding climatology back),
not residuals — otherwise numbers aren't comparable to published baselines.
With fake climatology (zeros), this is a no-op difference for now.

In [ ]:
def compute_metrics(pred_real, target_real, gl_weights,
                    horizon_names=['t+1h', 't+2h', 't+3h']):
    """
    pred_real / target_real: [B, n_horizons, H, W] — real TEC values (not residuals)
    Returns: {'rmse_t+1h': float, 'rmse_t+2h': float, 'rmse_t+3h': float}
    """
    metrics = {}
    sq_err  = (pred_real - target_real) ** 2 * gl_weights
    for i, name in enumerate(horizon_names):
        metrics[f'rmse_{name}'] = sq_err[:, i].mean().sqrt().item()
    return metrics

## Cell 14 — Training loop

Trains one epoch. For each batch:
1. Forward pass → residual predictions
2. Compute storm-weighted MSE + Sobolev loss
3. Backward pass + gradient clip + optimizer step

Validation adds fake climatology back (zeros) before computing RMSE,
so metrics are in real TECU units. Swap climatology tensor when Person 1 delivers.

In [ ]:
def train_one_epoch(model, loader, optimizer, gl_weights,
                    sobolev_weights, storm_bin_edges, storm_weights_table,
                    epoch, config, device):
    model.train()
    total_loss = total_mse = total_sob = 0.0
    n_batches = len(loader)

    for i, batch in enumerate(loader):
        tec_input  = batch["tec_input"].to(device)
        omni_input = batch["omni_input"].to(device)
        target     = batch["target"].to(device)
        kp         = omni_input[:, -1, 5].contiguous()
        sw         = lookup_storm_weight(kp, storm_bin_edges.to(device), storm_weights_table.to(device))

        optimizer.zero_grad()
        pred = model(tec_input, omni_input)

        loss, mse_val, sob_val = tec_loss(
            pred, target, gl_weights, model, sobolev_weights, sw, config['lambda_sobolev']
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_mse  += mse_val
        total_sob  += sob_val

        if i % 100 == 0:
            print(f'  batch {i}/{n_batches} | loss={loss.item():.4f}')

    n = len(loader)
    return {
        'train/loss':         total_loss / n,
        'train/mse':          total_mse  / n,
        'train/sobolev':      total_sob  / n,
        'train/teacher_prob': teacher_forcing_prob(epoch, config['n_epochs']),
    }

## Cell 15 — Autoregressive inference loop

Feeds model predictions back as input for forecasts beyond t+3h.
Operates in residual space internally, adds climatology back at the end
so the output is real TEC values ready for PyVista globe visualization.

In [ ]:
@torch.no_grad()
def autoregressive_rollout(model, tec_residual_history, omni_history,
                            climatology, n_steps, device):
    """
    tec_residual_history: [B, T, H, W]  — initial TEC residual history
    omni_history:         [B, T, 5]     — initial OMNI history
    climatology:          [8784, H, W]  — zeros placeholder
    n_steps:              int           — hours to forecast

    Returns: [B, n_steps, H, W] — real TEC, ready for PyVista globe
    """
    model.eval()
    tec_window  = tec_residual_history.clone().to(device)
    omni_window = omni_history.clone().to(device)
    forecasts   = []

    for _ in range(n_steps):
        pred      = model(tec_window, omni_window)             # [B, 3, H, W]
        next_step = pred[:, 0:1]                               # take t+1h
        forecasts.append(next_step)
        tec_window  = torch.cat([tec_window[:, 1:], next_step], dim=1)
        omni_window = torch.cat([omni_window[:, 1:], omni_window[:, -1:]], dim=1)

    residuals = torch.cat(forecasts, dim=1)                    # [B, n_steps, H, W]
    clim      = climatology[0].unsqueeze(0).unsqueeze(0).to(device)
    return residuals + clim                                    # real TEC

## Cell 16 — Baselines

Four baselines to beat:
- **Persistence:** TEC(t+k) = TEC(t). Minimum bar — just repeat the last observation.
- **Climatology:** Predict zero residual = predict the long-term mean. Stub until Person 1 delivers real climatology.
- **OMNI-only GRU:** Predict from solar wind alone — tests whether spatial TEC input matters.
- **Flat CNN:** Same architecture without SHT — tests whether spherical geometry actually helps.

In [ ]:
class PersistenceBaseline:
    """Repeat last observed TEC. Minimum meaningful bar."""
    def predict(self, tec_input):
        return tec_input[:, -1:].expand(-1, 3, -1, -1)   # [B, 3, H, W]


class ClimatologyBaseline:
    """
    Predicts zero residual = predicts the climatological mean.
    In residual mode this is a meaningful baseline, not trivially zero output.
    Stub until Person 1 delivers real climatology tensor.
    """
    def predict(self, tec_input):
        B, T, H, W = tec_input.shape
        return torch.zeros(B, 3, H, W, device=tec_input.device)


class OMNIOnlyGRU(nn.Module):
    """Predict from solar wind alone — no TEC spatial input."""
    def __init__(self, config):
        super().__init__()
        H, W          = config['H'], config['W']
        gru_h         = config['gru_hidden_size']
        self.n_horiz  = config['n_horizons']
        self.H, self.W = H, W
        self.gru  = nn.GRU(config['n_omni_features'], gru_h, batch_first=True)
        self.head = nn.Linear(gru_h, self.n_horiz * H * W)

    def forward(self, omni_input):
        _, h_n = self.gru(omni_input)
        return self.head(h_n.squeeze(0)).view(-1, self.n_horiz, self.H, self.W)


class FlatCNNBaseline(nn.Module):
    """Standard 2D CNN — tests whether spherical geometry helps."""
    def __init__(self, config):
        super().__init__()
        n_in    = config['n_timesteps']
        n_horiz = config['n_horizons']
        self.net = nn.Sequential(
            nn.Conv2d(n_in, 64,     3, padding=1), nn.GELU(),
            nn.Conv2d(64,  128,    3, padding=1), nn.GELU(),
            nn.Conv2d(128, 128,    3, padding=1), nn.GELU(),
            nn.Conv2d(128, 64,     3, padding=1), nn.GELU(),
            nn.Conv2d(64,  n_horiz, 1),
        )

    def forward(self, tec_input):
        return self.net(tec_input)


print('Baselines defined.')

Baselines defined.


## Cell 17 — Evaluate baselines

Run all four baselines and print RMSE per horizon.
All metrics in real TECU (climatology added back).
These are the numbers your SFNO model needs to beat.

In [ ]:
@torch.no_grad()
def evaluate_baselines(loader, gl_weights, climatology, config, device):
    persistence = PersistenceBaseline()
    clim_base   = ClimatologyBaseline()
    omni_gru    = OMNIOnlyGRU(config).to(device).eval()
    flat_cnn    = FlatCNNBaseline(config).to(device).eval()

    results = {'persistence': [], 'climatology': [], 'omni_only': [], 'flat_cnn': []}

    for batch in loader:
        tec_input       = batch["tec_input"]
        omni_input      = batch["omni_input"]
        target_residual = batch["target"]
        tec_input       = tec_input.to(device)
        omni_input      = omni_input.to(device)
        target_residual = target_residual.to(device)
        clim_slice      = climatology[0].unsqueeze(0).unsqueeze(0).to(device)
        target_real     = target_residual + clim_slice

        results['persistence'].append(
            compute_metrics(persistence.predict(tec_input) + clim_slice, target_real, gl_weights))
        results['climatology'].append(
            compute_metrics(clim_base.predict(tec_input) + clim_slice, target_real, gl_weights))
        results['omni_only'].append(
            compute_metrics(omni_gru(omni_input) + clim_slice, target_real, gl_weights))
        results['flat_cnn'].append(
            compute_metrics(flat_cnn(tec_input) + clim_slice, target_real, gl_weights))

    def avg(lst):
        return {k: float(np.mean([r[k] for r in lst])) for k in lst[0]}

    return {name: avg(res) for name, res in results.items()}

## Cell 18 — Checkpointing

Saves and loads model checkpoints. Best model by val loss is saved automatically during training.

In [ ]:
def save_checkpoint(model, optimizer, epoch, val_loss, path='best_model.pt'):
    torch.save({
        'epoch':     epoch,
        'model':     model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'val_loss':  val_loss,
        'config':    CONFIG,
    }, path)
    print(f'  ✓ Saved checkpoint (epoch {epoch}, val_loss={val_loss:.4f})')


def load_checkpoint(model, optimizer, path='best_model.pt'):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    print(f'  ✓ Loaded checkpoint from epoch {ckpt["epoch"]}')
    return ckpt['epoch'], ckpt['val_loss']


print('Checkpointing functions ready.')

Checkpointing functions ready.


## Cell 19 — Main: run everything

Runs the full pipeline:
1. Build dataloaders
2. Evaluate baselines
3. Train SFNO model for n_epochs
4. Log to W&B
5. Autoregressive rollout smoke test

W&B will ask you to log in first time — follow the link it prints.

In [ ]:
# Build everything
train_loader = make_dataloader(CONFIG, split="train")
val_loader   = make_dataloader(CONFIG, split="val")
climatology  = make_fake_climatology(CONFIG)   # zeros -- swap with Person 1's tensor

model     = TECSFNOModel(CONFIG).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3,
)

print('=' * 55)
print('TEC SFNO -- Residual, Real Data')
print('=' * 55)
print(f"Grid:    {CONFIG['H']}x{CONFIG['W']} Gauss-Legendre")
print(f"L_max:   {CONFIG['l_max']}")
print(f"TEC:     residual mode")
print(f"Device:  {DEVICE}")
print(f"Params:  {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# W&B
wandb.init(project=CONFIG['wandb_project'], name=CONFIG['wandb_run_name'], config=CONFIG)

# Baselines
print('\nBaselines (beat these):')
for name, metrics in evaluate_baselines(val_loader, gl_weights, climatology, CONFIG, DEVICE).items():
    print(f'  {name:15s}: { {k: f"{v:.4f}" for k, v in metrics.items()} }')

# Training
best_val_loss = float('inf')
print('\nTraining...')

for epoch in range(CONFIG['n_epochs']):
    train_m = train_one_epoch(
        model, train_loader, optimizer,
        gl_weights, sobolev_weights, storm_bin_edges, storm_weights_table,
        epoch, CONFIG, DEVICE
    )
    val_m = validate(
        model, val_loader, gl_weights, sobolev_weights, storm_bin_edges, storm_weights_table,
        climatology, CONFIG, DEVICE
    )
    scheduler.step(val_m['val/loss'])

    wandb.log({**train_m, **val_m, 'epoch': epoch,
               'lr': optimizer.param_groups[0]['lr']}, step=epoch)

    print(
        f"Epoch {epoch:03d} | "
        f"train={train_m['train/loss']:.4f} | "
        f"val={val_m['val/loss']:.4f} | "
        f"rmse_t+1h={val_m['rmse_t+1h']:.4f} | "
        f"tf={train_m['train/teacher_prob']:.2f}"
    )

    if val_m['val/loss'] < best_val_loss:
        best_val_loss = val_m['val/loss']
        save_checkpoint(model, optimizer, epoch, best_val_loss)
        wandb.save('best_model.pt')
        import shutil
        shutil.copy('best_model.pt', '/content/drive/MyDrive/tec_data/best_model.pt')
        print(f'  -> backed up to Drive')

# Rollout smoke test
print('\nRollout test (12 steps)...')
_batch = next(iter(val_loader))
_tec, _omni = _batch["tec_input"].to(DEVICE), _batch["omni_input"].to(DEVICE)
forecast = autoregressive_rollout(
    model, _tec, _omni, climatology, n_steps=12, device=DEVICE
)
print(f'  Output shape: {forecast.shape}   (expected [B, 12, H, W] -- real TEC for PyVista)')

wandb.finish()
print('\nDone.')


TEC SFNO -- Residual, Real Data
Grid:    23x45 Gauss-Legendre
L_max:   23
TEC:     residual mode
Device:  cuda
Params:  52,316,547


epoch,▁▅█
lr,▁▁▁
rmse_t+1h,█▁▁
rmse_t+2h,█▃▁
rmse_t+3h,█▃▁
train/loss,█▁▁
train/mse,█▂▁
train/sobolev,█▁▁
train/teacher_prob,█▅▁
val/loss,█▂▁
epoch,2



Baselines (beat these):
  persistence    : {'rmse_t+1h': '0.0539', 'rmse_t+2h': '0.1028', 'rmse_t+3h': '0.1463'}
  climatology    : {'rmse_t+1h': '0.1781', 'rmse_t+2h': '0.1781', 'rmse_t+3h': '0.1781'}
  omni_only      : {'rmse_t+1h': '0.1789', 'rmse_t+2h': '0.1790', 'rmse_t+3h': '0.1791'}
  flat_cnn       : {'rmse_t+1h': '0.1780', 'rmse_t+2h': '0.1786', 'rmse_t+3h': '0.1843'}

Training...
  batch 0/1721 | loss=0.3843
  batch 100/1721 | loss=0.1480
  batch 200/1721 | loss=0.0677
  batch 300/1721 | loss=0.0481
  batch 400/1721 | loss=0.0481
  batch 500/1721 | loss=0.0239
  batch 600/1721 | loss=0.0118
  batch 700/1721 | loss=0.0082
  batch 800/1721 | loss=0.0247
  batch 900/1721 | loss=0.0073
  batch 1000/1721 | loss=0.0074
  batch 1100/1721 | loss=0.0172
  batch 1200/1721 | loss=0.0050
  batch 1300/1721 | loss=0.0053
  batch 1400/1721 | loss=0.0051
  batch 1500/1721 | loss=0.0052
  batch 1600/1721 | loss=0.0331
  batch 1700/1721 | loss=0.0054
Epoch 000 | train=0.0352 | val=0.0030 | rm

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


  ✓ Saved checkpoint (epoch 0, val_loss=0.0030)
  -> backed up to Drive
  batch 0/1721 | loss=0.0086
  batch 100/1721 | loss=0.0090
  batch 200/1721 | loss=0.0055
  batch 300/1721 | loss=0.0048
  batch 400/1721 | loss=0.0062
  batch 500/1721 | loss=0.0062
  batch 600/1721 | loss=0.0050
  batch 700/1721 | loss=0.0046
  batch 800/1721 | loss=0.0051
  batch 900/1721 | loss=0.0043
  batch 1000/1721 | loss=0.0051
  batch 1100/1721 | loss=0.0031
  batch 1200/1721 | loss=0.0077
  batch 1300/1721 | loss=0.0124
  batch 1400/1721 | loss=0.0055
  batch 1500/1721 | loss=0.0040
  batch 1600/1721 | loss=0.0037
  batch 1700/1721 | loss=0.0054
Epoch 001 | train=0.0072 | val=0.0023 | rmse_t+1h=0.0297 | tf=0.95
  ✓ Saved checkpoint (epoch 1, val_loss=0.0023)
  -> backed up to Drive
  batch 0/1721 | loss=0.0044
  batch 100/1721 | loss=0.0059
  batch 200/1721 | loss=0.0045
  batch 300/1721 | loss=0.0045
  batch 400/1721 | loss=0.0042
  batch 500/1721 | loss=0.0049
  batch 600/1721 | loss=0.0346
  batch 70